# Electron Orbital ZBW Mixing Fractions (N_k=1) – Monte Carlo Refinement

This notebook refines the mixing fractions for the electron (minimal cage, N_k=1) using Monte Carlo sampling over lattice paths and thermal fluctuations. Goal: tighter bound on qDP/hDP fractions → better δμ_e estimate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Golden ratio and base gradients (from previous code)
phi = (1 + np.sqrt(5)) / 2
base_gradient = 1.0
gradients = {
    'eDP': (phi - 1) * base_gradient,  # Favorable (lowest energy)
    'hDP': base_gradient,
    'qDP': phi * base_gradient         # Least favorable
}

N_k = 1.0  # Electron minimal cage
T = N_k    # Thermal scale
n_samples = 50000  # MC samples (increase for precision)

# Add lattice noise (simulating finite-size effects)
noise_scale = 0.03  # small perturbation from 120-vertex lattice
noise = np.random.normal(0, noise_scale, (n_samples, 3))
energies = np.array([gradients['eDP'], gradients['hDP'], gradients['qDP']]) + noise

# Boltzmann probabilities
probs = np.exp(-energies / T)
probs /= probs.sum(axis=1)[:, np.newaxis]

# Average fractions
avg_fracs = probs.mean(axis=0)
std_fracs = probs.std(axis=0)
labels = ['eDP', 'hDP', 'qDP']

print("Refined average fractions (electron, N_k=1):")
for label, avg, std in zip(labels, avg_fracs, std_fracs):
    print(f"{label}: {avg:.5f} ± {std:.5f}")

# Plot histogram of qDP fraction (key for g-2)
plt.hist(probs[:, 2], bins=100, density=True, alpha=0.7, color='orange', label='qDP')
plt.axvline(avg_fracs[2], color='red', linestyle='--', label=f'Mean qDP = {avg_fracs[2]:.5f}')
plt.xlabel('qDP Fraction')
plt.ylabel('Density')
plt.title('Monte Carlo Distribution of qDP Fraction (Electron, N_k=1)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../figures/electron-qdp-distribution.png')
plt.show()

## Preliminary g-2 Estimate
Using refined qDP fraction and previous formula (C ≈ 9.6e-7, S = α/(2π)).
Raw boost ≈ C × (f_qDP + 0.7 × f_hDP)

In [ ]:
C = 9.6e-7
alpha = 1 / 137.035999084
S = alpha / (2 * np.pi)

f_qDP = avg_fracs[2]
f_hDP = avg_fracs[1]
mixing_sum = f_qDP + 0.7 * f_hDP
raw_delta = C * mixing_sum
delta_mu_e = raw_delta * S

print(f"Raw mixing boost: {raw_delta:.2e}")
print(f"Suppression S: {S:.4e}")
print(f"Electron δμ estimate: {delta_mu_e:.2e}")
print(f"Current experimental bound: < 10^{-12}")